In [1]:
!pip install datasets

In [2]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer
import numpy as np
import random
import torch
import os
from datasets import load_dataset, Dataset
import time
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
ds = load_dataset('NLBSE/nlbse27-code-comment-classification')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
tokenizer = RobertaTokenizer.from_pretrained("NLBSE/nlbse27-code-comment-classification-baseline")

total_flops = 0
total_time = 0
scores = []


test = ds["test"]

model = RobertaForSequenceClassification.from_pretrained("NLBSE/nlbse27-code-comment-classification-baseline")
model.to(device)
model.eval()

predictions = []

test_data = test.to_pandas()

# Define batch size
batch_size = 600  # Adjust based on your GPU memory

# Tokenize all data at once
tokenized_data = tokenizer(
    test_data["Function"].tolist(),
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=512
)

# Create a DataLoader for batch processing
dataset = TensorDataset(tokenized_data["input_ids"], tokenized_data["attention_mask"])
data_loader = DataLoader(dataset, batch_size=batch_size)

# Initialize variables
predictions = []
pbar = tqdm(total=len(test_data), desc=f"Predicting...")

# Process batches
for batch in data_loader:
    input_ids, attention_mask = batch
    input_ids = input_ids.to(device, non_blocking=True)
    attention_mask = attention_mask.to(device, non_blocking=True)

    # Perform forward pass and measure performance
    begin = time.time()
    with torch.no_grad():
      with torch.cuda.amp.autocast():
        with torch.profiler.profile(with_flops=True) as p:
            for _ in range(2):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    total = time.time() - begin
    total_time = total_time + total

    # Calculate FLOPs for the batch
    total_flops = total_flops + (sum(k.flops for k in p.key_averages()) / 1e9)

    # Extract logits and compute predictions
    logits = outputs.logits
    probs = torch.sigmoid(logits)
    threshold = 0.5
    batch_predictions = (probs > threshold).int().cpu().numpy().tolist()
    predictions.extend(batch_predictions)


    del input_ids, attention_mask, outputs, logits, probs
    torch.cuda.empty_cache()

    # Update progress bar
    pbar.update(len(batch_predictions))

pbar.close()


labels = torch.tensor(np.array(test_data["Label"].tolist())).to(device)
labels = labels.cpu().numpy()

labels_name = ["Vulnerability", "MAT"]
num_classes = len(labels_name)
class_metrics = {}


predictions = np.array(predictions)
labels = np.array(labels)

for class_idx in range(num_classes):
    tp = np.sum((predictions[:, class_idx] == 1) & (labels[:, class_idx] == 1))
    fp = np.sum((predictions[:, class_idx] == 1) & (labels[:, class_idx] == 0))
    tn = np.sum((predictions[:, class_idx] == 0) & (labels[:, class_idx] == 0))
    fn = np.sum((predictions[:, class_idx] == 0) & (labels[:, class_idx] == 1))

    # Calculate precision, recall, and F1 score for the current class
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    # Store metrics for the current class
    class_metrics[f"class_{labels_name[class_idx]}_true_positives"] = tp
    class_metrics[f"class_{labels_name[class_idx]}_false_positives"] = fp
    class_metrics[f"class_{labels_name[class_idx]}_true_negatives"] = tn
    class_metrics[f"class_{labels_name[class_idx]}_false_negatives"] = fn
    class_metrics[f"class_{labels_name[class_idx]}_precision"] = precision
    class_metrics[f"class_{labels_name[class_idx]}_recall"] = recall
    class_metrics[f"class_{labels_name[class_idx]}_f1"] = f1
    scores.append({'cat': labels_name[class_idx],'precision': precision,'recall': recall,'f1': f1})

print("----------------RESULTS--------------------")

print("Compute in GFLOPs:", total_flops/2)
print("Avg runtime in seconds:", total_time/2)
scores = pd.DataFrame(scores)
scores

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading...:  35%|███▌      | 10800/30655 [03:17<06:03, 54.65it/s]
/tmp/ipykernel_4785/1840351287.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():

Predicting...: 100%|██████████| 30655/30655 [07:07<00:00, 71.69it/s]

----------------RESULTS--------------------
Compute in GFLOPs: 2666509.7131617293
Avg runtime in seconds: 198.93055331707


,cat,precision,recall,f1
0,Vulnerability,0.942651,0.935336,0.938979
1,MAT,0.978261,0.801980,0.881393


In [8]:
max_avg_runtime = 1000
max_avg_flops = 3000000
# s𝑢𝑏𝑚𝑖𝑠𝑠𝑖𝑜𝑛_𝑠𝑐𝑜𝑟𝑒(𝑚𝑜𝑑𝑒𝑙)=(𝑎𝑣𝑔. 𝐹1)×0.60+max(((𝑚𝑎𝑥_𝑎𝑣𝑔_𝑟𝑢𝑛𝑡𝑖𝑚𝑒−𝑚𝑒𝑎𝑠𝑢𝑟𝑒𝑑_𝑎𝑣𝑔_𝑟𝑢𝑛𝑡𝑖𝑚𝑒)/𝑚𝑎𝑥_𝑎𝑣𝑔_𝑟𝑢𝑛𝑡𝑖𝑚𝑒),0)×0.2+max((𝑚𝑎𝑥_GFLOPs−𝑚𝑒𝑎𝑠𝑢𝑟𝑒𝑑_GFLOPs)/𝑚𝑎𝑥_GFLOPs),0)×0.2
def score(avg_f1, avg_runtime, avg_flops):
    return (
        0.6 * avg_f1
        + 0.2 * max((max_avg_runtime - avg_runtime) / max_avg_runtime, 0)
        + 0.2 * max((max_avg_flops - avg_flops) / max_avg_flops, 0)
    )


avg_f1 = scores.f1.mean()
avg_p = scores.precision.mean()
avg_r = scores.recall.mean()
avg_runtime = total_time/2
avg_flops = total_flops/2

print(avg_p)
print(avg_r)
print(avg_f1)
print(avg_runtime)
print(avg_flops)

round(score(avg_f1, avg_runtime, avg_flops), 2)

0.9604558809426247
0.8686582247560212
0.9101860725779012
198.93055331707
2666509.7131617293


np.float64(0.71)